# Short-Horizon Forecast Review

This notebook reviews one-month-ahead and three-month sequential forecasts from the first held-out test origin. Each model uses the same monthly feature contract and updates demand-derived lag and rolling features recursively after each predicted month.


## Forecast origin and recursive protocol

The forecast origin is the start of the test period. Calendar and lagged driver features come from the shared feature matrix, while future demand lags and rolling values are updated from earlier forecasts.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.horizon_forecasting import save_horizon_forecasts

saved_paths = save_horizon_forecasts(PROJECT_ROOT, horizon=3)
forecasts = pd.read_csv(saved_paths['forecasts'])
display(forecasts.round(3))
print(saved_paths)


In [ ]:
colors = {'Seasonal naive': '#6b7280', 'Random Forest': '#2f6f9f', 'ANN': '#d9480f', 'LSTM': '#2a9d8f'}
figure, axis = plt.subplots(figsize=(12, 6))
for model, group in forecasts.groupby('model'):
    axis.plot(group['horizon_step'], group['forecast_m3'], marker='o', color=colors[model], label=model)
actual = forecasts.drop_duplicates('horizon_step')
axis.plot(actual['horizon_step'], actual['actual_m3'], marker='o', color='#111827', linewidth=2.5, label='Actual')
axis.set_xticks([1, 2, 3])
axis.set_xlabel('Months ahead from test origin')
axis.set_ylabel('Residential water demand (m3)')
axis.set_title('One-month and three-month sequential forecasts')
axis.legend()
figure.tight_layout()
plt.show()
